In [1]:
# Series & DataFrame

In [22]:
# Series - one metric across fleet of servers
import pandas as pd
cpu = pd.Series(
    [45,78,30,92,55], 
    index = ["srv-01", "srv-02", "srv-03", "srv-04", "srv-05"],
    name="cpu_pct"
)
print(cpu)

srv-01    45
srv-02    78
srv-03    30
srv-04    92
srv-05    55
Name: cpu_pct, dtype: int64


srv-01    45
srv-02    78
srv-03    30
srv-04    92
srv-05    55
Name: cpu_pct, dtype: int64

In [3]:
# Series operations - instant servers health
print(cpu.mean())
print(cpu.max())
print(cpu.min())
print(cpu.std())
print(cpu.sum())
print(cpu[cpu>80])

60.0
92
30
24.9899979991996
300
srv-04    92
Name: cpu_pct, dtype: int64


In [26]:
# DataFrames - full server snapshot
import pandas as pd
data = {
    "server_id": ["srv-01", "srv-02", "srv-03", "srv-04", "srv-05"],
    "cpu_pct" : [45,78,30,92,55],
    "memory_pct" : [60,82,40,95,65],
    "response_ms" : [120,340,89, 950, 210],
    "disk_pct" : [55,70,45,88,60],
    "status" : ["ok", "warning", "ok", "critical", "ok"]
}
df = pd.DataFrame(data)
print(df)
# Accessing columns
print(df["cpu_pct"]) # single column - series
print(df[["server_id", "disk_pct", "status"]]) # multiple columns - DataFrame
#Key attributes
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.index)
print(len(df))

  server_id  cpu_pct  memory_pct  response_ms  disk_pct    status
0    srv-01       45          60          120        55        ok
1    srv-02       78          82          340        70   warning
2    srv-03       30          40           89        45        ok
3    srv-04       92          95          950        88  critical
4    srv-05       55          65          210        60        ok
0    45
1    78
2    30
3    92
4    55
Name: cpu_pct, dtype: int64
  server_id  disk_pct    status
0    srv-01        55        ok
1    srv-02        70   warning
2    srv-03        45        ok
3    srv-04        88  critical
4    srv-05        60        ok
(5, 6)
Index(['server_id', 'cpu_pct', 'memory_pct', 'response_ms', 'disk_pct',
       'status'],
      dtype='object')
server_id      object
cpu_pct         int64
memory_pct      int64
response_ms     int64
disk_pct        int64
status         object
dtype: object
RangeIndex(start=0, stop=5, step=1)
5


In [5]:
# Derived columns
print(df)
df["overloaded"] = (df["cpu_pct"] > 80) & (df["memory_pct"] > 80)
df["load_score"] = (df["cpu_pct"] + df["memory_pct"]) / 2
df["response_sec"] = df["response_ms"] / 1000

print(df[["server_id", "overloaded", "load_score", "response_sec"]])

  server_id  cpu_pct  memory_pct  response_ms  disk_pct    status
0    srv-01       45          60          120        55        ok
1    srv-02       78          82          340        70   warning
2    srv-03       30          40           89        45        ok
3    srv-04       92          95          950        88  critical
4    srv-05       55          65          210        60        ok
  server_id  overloaded  load_score  response_sec
0    srv-01       False        52.5         0.120
1    srv-02       False        80.0         0.340
2    srv-03       False        35.0         0.089
3    srv-04        True        93.5         0.950
4    srv-05       False        60.0         0.210


In [6]:
# Drop columns
df = df.drop(columns=["response_sec"])
print(df)

  server_id  cpu_pct  memory_pct  response_ms  disk_pct    status  overloaded  \
0    srv-01       45          60          120        55        ok       False   
1    srv-02       78          82          340        70   warning       False   
2    srv-03       30          40           89        45        ok       False   
3    srv-04       92          95          950        88  critical        True   
4    srv-05       55          65          210        60        ok       False   

   load_score  
0        52.5  
1        80.0  
2        35.0  
3        93.5  
4        60.0  


In [7]:
# Set and reset index
df = df.set_index("server_id") # server_id becomes the index (primary key for refenence)
print(df.loc["srv-04"])

cpu_pct              92
memory_pct           95
response_ms         950
disk_pct             88
status         critical
overloaded         True
load_score         93.5
Name: srv-04, dtype: object


In [43]:
# dtype fix and copy ()
df["cpu_pct"] = df["cpu_pct"].astype(float)
df_clean = df.copy()
df_features = df.copy()
print(df)
print(df_clean)
print(df_features)  #output looks same but their ids are different, which can be checked using id().

  server_id  cpu_pct  memory_pct  response_ms  disk_pct    status
0    srv-01     45.0          60          120        55        ok
1    srv-02     78.0          82          340        70   warning
2    srv-03     30.0          40           89        45        ok
3    srv-04     92.0          95          950        88  critical
4    srv-05     55.0          65          210        60        ok
  server_id  cpu_pct  memory_pct  response_ms  disk_pct    status
0    srv-01     45.0          60          120        55        ok
1    srv-02     78.0          82          340        70   warning
2    srv-03     30.0          40           89        45        ok
3    srv-04     92.0          95          950        88  critical
4    srv-05     55.0          65          210        60        ok
  server_id  cpu_pct  memory_pct  response_ms  disk_pct    status
0    srv-01     45.0          60          120        55        ok
1    srv-02     78.0          82          340        70   warning
2    srv-0

In [9]:
# DataFrame from list of dicts - API Ingestion pattern
incidents = [
    {"ticket_id" : "INC001", "server": "srv-04", "category" : "CPU Hight", "priority" : 1, "resolved": False},
    {"ticket_id" : "INC002", "server" : "srv-02", "category" : "Disk Full", "priority" : 2, "resolved": False},
    {"ticket_id" : "INC003", "server" : "srv-01", "category" : "Network Lag", "priority" : 3, "resolved" : True},
    {"ticket_id" : "INC004", "server" : "srv-04", "category" : "Memory Leak", "priority" : 1, "resolved": False},
]
df_inc = pd.DataFrame(incidents)
print(df_inc)

  ticket_id  server     category  priority  resolved
0    INC001  srv-04    CPU Hight         1     False
1    INC002  srv-02    Disk Full         2     False
2    INC003  srv-01  Network Lag         3      True
3    INC004  srv-04  Memory Leak         1     False


In [53]:
# Datasets generation to work on: 
# generating server metrics and saving to csv file
# generating incident tickets and saving to csv file
# generating Appending logs and saving to csv file 
import pandas as pd
import numpy as np

np.random.seed(42)

# --- Dataset 1: Server Metrics ---
servers     = ["srv-01", "srv-02", "srv-03", "srv-04", "srv-05"]
timestamps  = pd.date_range("2026-01-01", periods=100, freq="1h")

records = []
for ts in timestamps:
    for srv in servers:
        records.append({
            "timestamp":   ts,
            "server_id":   srv,
            "cpu_pct":     round(np.random.uniform(20, 99), 1),
            "memory_pct":  round(np.random.uniform(30, 98), 1),
            "response_ms": round(np.random.uniform(50, 1200), 1),
            "disk_pct":    round(np.random.uniform(30, 95), 1),
            "status":      np.random.choice(["ok","warning","critical"], p=[0.75, 0.20, 0.05])
        })

df_metrics = pd.DataFrame(records)
df_metrics.to_csv("server_metrics.csv", index=False)
print("server_metrics.csv saved →", df_metrics.shape)


# --- Dataset 2: Incident Tickets ---
categories = ["CPU High", "Disk Full", "Network Lag", "Memory Leak", "Service Down"]
priorities = [1, 2, 3]

tickets = []
for i in range(200):
    created = pd.Timestamp("2026-01-01") + pd.Timedelta(hours=np.random.randint(0, 2400))
    resolved = created + pd.Timedelta(hours=np.random.randint(1, 72))
    is_resolved = np.random.choice([True, False], p=[0.65, 0.35])
    tickets.append({
        "ticket_id":   f"INC{str(i+1).zfill(4)}",
        "server_id":   np.random.choice(servers),
        "category":    np.random.choice(categories),
        "priority":    np.random.choice(priorities, p=[0.3, 0.5, 0.2]),
        "created_at":  created,
        "resolved_at": resolved if is_resolved else pd.NaT,
        "resolved":    is_resolved
    })

df_tickets = pd.DataFrame(tickets)
df_tickets.to_csv("incidents.csv", index=False)
print("incidents.csv saved →", df_tickets.shape)


# --- Dataset 3: Application Logs ---
log_levels = ["INFO", "WARNING", "ERROR", "CRITICAL"]
messages = [
    "CPU threshold exceeded",
    "Disk usage at 90%",
    "Connection timeout error",
    "Memory allocation failed",
    "Service restarted successfully",
    "Network packet loss detected",
    "Database query slow",
    "Authentication failed",
]

logs = []
for i in range(300):
    ts = pd.Timestamp("2026-01-01") + pd.Timedelta(minutes=np.random.randint(0, 14400))
    logs.append({
        "timestamp":   ts,
        "server_id":   np.random.choice(servers),
        "log_level":   np.random.choice(log_levels, p=[0.5, 0.25, 0.15, 0.10]),
        "message":     np.random.choice(messages),
        "error_code":  np.random.choice(["E001","E002","E003","E004", None], p=[0.2,0.2,0.2,0.2,0.2])
    })

df_logs = pd.DataFrame(logs)
df_logs.to_csv("app_logs.csv", index=False)
print("app_logs.csv saved →", df_logs.shape)

server_metrics.csv saved → (500, 7)
incidents.csv saved → (200, 7)
app_logs.csv saved → (300, 5)


In [54]:
# Loading server_metrics.csv into dataframe called df_metrics and prints its shape.
import pandas as pd

# Load CSV into DataFrame
# df_metrics = pd.read_csv("server_metrics.csv")
df_metrics = pd.read_csv("server_metrics.csv", parse_dates = ["timestamp"])

# Print shape (rows, columns)
print(df_metrics.shape)
print(df_metrics.dtypes)
print(df_metrics)

(500, 7)
timestamp      datetime64[ns]
server_id              object
cpu_pct               float64
memory_pct            float64
response_ms           float64
disk_pct              float64
status                 object
dtype: object
              timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
495 2026-01-05 03:00:00    srv-01     88.7        38.9        959.1      38.1   
496 2026-01-05 03:00:00    srv-02     41.8        89.6       1135.6      39.7   
497 2026-01-05 03:00:00    srv-03     

In [16]:
# create series called memory from memory_pct column of df_metrics. print the mean, max and min.
import numpy as np
import pandas as pd
memory = df_metrics["memory_pct"]
print(memory)
print(memory.mean())
print(memory.max())
print(memory.min())

0      94.6
1      33.9
2      96.0
3      50.7
4      39.5
       ... 
495    38.9
496    89.6
497    62.9
498    43.8
499    69.3
Name: memory_pct, Length: 500, dtype: float64
63.355
98.0
30.2


In [56]:
# Load incidents.csv into df_tickets. From it create new DataFrame called df_inc_raw using only these columns
# ticket_it, server_id, category, priority, resolved.
import pandas as pd
df_tickets = pd.read_csv("incidents.csv", parse_dates =["created_at", "resolved_at"])
print(df_tickets.shape)
df_inc_raw = df_tickets[["ticket_id", "server_id", "category", "priority", "resolved"]]
print(df_inc_raw.shape)
print(df_inc_raw.head())

(200, 7)
(200, 5)
  ticket_id server_id      category  priority  resolved
0   INC0001    srv-04  Service Down         2     False
1   INC0002    srv-01      CPU High         3      True
2   INC0003    srv-03  Service Down         2     False
3   INC0004    srv-02  Service Down         1     False
4   INC0005    srv-04     Disk Full         2      True


In [62]:
# Add a column is_high_piority to df_tickets - True where priority ==1, False otherwise.
df_tickets["is_high_priority"] = df_tickets["priority"] == 1
print(df_tickets["is_high_priority"].value_counts())
print(df_tickets[["ticket_id", "priority", "is_high_priority"]].head(10))

is_high_priority
False    150
True      50
Name: count, dtype: int64
  ticket_id  priority  is_high_priority
0   INC0001         2             False
1   INC0002         3             False
2   INC0003         2             False
3   INC0004         1              True
4   INC0005         2             False
5   INC0006         1              True
6   INC0007         2             False
7   INC0008         3             False
8   INC0009         2             False
9   INC0010         2             False


In [64]:
print((df_tickets).head(10))

  ticket_id server_id      category  priority          created_at  \
0   INC0001    srv-04  Service Down         2 2026-03-24 02:00:00   
1   INC0002    srv-01      CPU High         3 2026-03-19 15:00:00   
2   INC0003    srv-03  Service Down         2 2026-04-02 10:00:00   
3   INC0004    srv-02  Service Down         1 2026-03-13 20:00:00   
4   INC0005    srv-04     Disk Full         2 2026-03-18 09:00:00   
5   INC0006    srv-03   Network Lag         1 2026-01-29 23:00:00   
6   INC0007    srv-03      CPU High         2 2026-01-05 16:00:00   
7   INC0008    srv-05  Service Down         3 2026-03-07 12:00:00   
8   INC0009    srv-02      CPU High         2 2026-01-09 08:00:00   
9   INC0010    srv-05  Service Down         2 2026-01-18 05:00:00   

          resolved_at  resolved  is_high_priority  
0                 NaT     False             False  
1 2026-03-21 12:00:00      True             False  
2                 NaT     False             False  
3                 NaT     False 

In [65]:
# Add a column load_score to df_metrics = (cpu_pct + memory_pct) / 2
# print the top 5 rows showing only server_id, cpu_pct, memory_pct, load_score
df_metrics["load_score"] = (df_metrics["cpu_pct"] + df_metrics["memory_pct"]) / 2
print(df_metrics["load_score"].isnull().sum())
print(df_metrics[["server_id","cpu_pct", "memory_pct", "load_score"]].head())

0
  server_id  cpu_pct  memory_pct  load_score
0    srv-01     49.6        94.6        72.1
1    srv-02     32.3        33.9        33.1
2    srv-03     21.6        96.0        58.8
3    srv-04     34.5        50.7        42.6
4    srv-05     68.3        39.5        53.9


In [67]:
# Set ticket_id as the index of df_tickets. Look up ticket INC0042 using .loc. Then reset the index.
print(df_tickets.head(), "\n\n")
print("Index before: ", df_tickets.index.name)
df_tickets = df_tickets.set_index("ticket_id")
print("Index after:", df_tickets.index.name)
print(df_tickets.loc["INC0042"], "\n")
#print(df_tickets.head(), "\n")
df_tickets = df_tickets.reset_index()
print("Index reset:", df_tickets.index.name)
#print(df_tickets.head())

  ticket_id server_id      category  priority          created_at  \
0   INC0001    srv-04  Service Down         2 2026-03-24 02:00:00   
1   INC0002    srv-01      CPU High         3 2026-03-19 15:00:00   
2   INC0003    srv-03  Service Down         2 2026-04-02 10:00:00   
3   INC0004    srv-02  Service Down         1 2026-03-13 20:00:00   
4   INC0005    srv-04     Disk Full         2 2026-03-18 09:00:00   

          resolved_at  resolved  is_high_priority  
0                 NaT     False             False  
1 2026-03-21 12:00:00      True             False  
2                 NaT     False             False  
3                 NaT     False              True  
4 2026-03-21 01:00:00      True             False   


Index before:  None
Index after: ticket_id
server_id                        srv-05
category                       CPU High
priority                              3
created_at          2026-02-15 02:00:00
resolved_at                         NaT
resolved                   

In [44]:
# Make safe copy of df_metrics called df_features. Confirm they are separate objects using id()
df_features = df_metrics.copy()

print("df_metrics id: ", id(df_metrics))
print("df_features id : " , id(df_features))

df_metrics id:  1743673785376
df_features id :  1743724095072
